# Vehicle Insurance Purchase Prediction

**Classification Project 28 of 100** — Predict whether a customer will purchase vehicle insurance from a cross-sell campaign.

EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.


## 2. Project Overview

Predict whether existing health insurance customers will respond positively to a vehicle insurance cross-sell offer. This is a classic response modeling / propensity-to-buy problem.

Binary classification with class imbalance (most customers don't buy).


## 3. Learning Objectives

1. Cross-sell / response modeling as a classification problem
2. Working with insurance domain features (vehicle age, premium, channel)
3. Handling imbalanced binary classification
4. Feature encoding for mixed data types
5. Model comparison with LazyPredict and FLAML
6. Business metrics: lift, targeting efficiency


## 4. Problem Statement

> Given a customer's demographics, vehicle, and insurance history, predict whether they will purchase vehicle insurance.

Binary classification. ROC-AUC and F1 are primary metrics.


## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Insurance companies | Targeted cross-sell campaigns |
| Marketing teams | Reduce campaign waste |
| Sales agents | Focus on high-propensity leads |
| ML learners | Response modeling with mixed features |


## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | anmolkumar/health-insurance-cross-sell-prediction |
| Rows | ~380k (sampled to 50k) |
| Features | Age, gender, vehicle age, premium, region, channel |
| Target | Response (0/1) |
| Balance | Imbalanced (~12% positive) |


## 7. Dataset Source and License Notes

- Kaggle: anmolkumar/health-insurance-cross-sell-prediction
- Analytics Vidhya hackathon dataset
- License: Check Kaggle page for details.


## 8. Environment Setup


In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")


## 9. Imports


In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")


## 10. Configuration / Constants


In [ ]:
KAGGLE_SLUG = "anmolkumar/health-insurance-cross-sell-prediction"
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 90; MAX_ROWS = 50000
print(f"Kaggle slug: {KAGGLE_SLUG}")


## 11. Dataset Download or Loading


In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
train_files = [f for f in csv_files if 'train' in os.path.basename(f).lower()]
chosen = train_files[0] if train_files else sorted(csv_files, key=os.path.getsize, reverse=True)[0]
df_raw = pd.read_csv(chosen)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()


## 12. Data Validation Checks


In [ ]:
print(f"Missing values:\n{df_raw.isnull().sum()}\n")
df = df_raw.copy()
target_candidates = [c for c in df.columns if any(h in c.lower() for h in ['response','purchase','target','clicked'])]
TARGET = target_candidates[0] if target_candidates else df.columns[-1]
print(f"Target column: {TARGET}")
id_cols = [c for c in df.columns if c.lower() in ['id','index','unnamed: 0']]
if id_cols: df = df.drop(columns=id_cols); print(f"Dropped: {id_cols}")
dupes = df.duplicated().sum()
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"Sampled to {MAX_ROWS}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")


## 13. Exploratory Data Analysis


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=['steelblue','coral'])
ax.set_title(f'Target Distribution: {TARGET}')
plt.tight_layout(); plt.show()


In [ ]:
num_feats = df.select_dtypes(include=[np.number]).columns.tolist()
num_feats = [c for c in num_feats if c != TARGET]
if len(num_feats) >= 4:
    fig, axes = plt.subplots(2, 2, figsize=(12,8))
    for ax, col in zip(axes.flat, num_feats[:4]):
        for val in sorted(df[TARGET].unique()):
            ax.hist(df[df[TARGET]==val][col].dropna(), bins=25, alpha=0.4, label=f'{TARGET}={val}')
        ax.set_title(col); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()


In [ ]:
cat_feats = df.select_dtypes(include=['object','category']).columns.tolist()
if cat_feats:
    for col in cat_feats[:4]:
        print(f"{col}: {df[col].nunique()} unique")
print(f"\nNumeric: {len(num_feats)}, Categorical: {len(cat_feats)}")


## 14. Target Analysis

Binary target with imbalance — most customers don't respond. ROC-AUC is more informative than accuracy.


In [ ]:
print(df[TARGET].value_counts(normalize=True))
print(f"\nPositive response rate: {df[TARGET].mean():.1%}")


## 15. Train / Validation / Test Split


In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
if y.dtype == object:
    le = LabelEncoder(); y = pd.Series(le.fit_transform(y), name=TARGET)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")


## 16. Preprocessing Strategy

- Numeric: impute median, standardize
- Categorical: impute mode, one-hot encode


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
transformers = []
if num_cols: transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc', StandardScaler())]), num_cols))
if cat_cols: transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')


## 17. Feature Engineering

Create domain-specific features for insurance cross-sell.


In [ ]:
def engineer_features(d):
    d = d.copy()
    nc = d.select_dtypes(include=[np.number]).columns
    premium_cols = [c for c in nc if 'premium' in c.lower()]
    age_cols = [c for c in nc if c.lower() == 'age']
    if premium_cols and age_cols:
        d['PREMIUM_PER_AGE'] = d[premium_cols[0]] / d[age_cols[0]].clip(lower=18)
    return d
X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
transformers = []
if num_cols: transformers.append(('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num_cols))
if cat_cols: transformers.append(('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')


## 18. Baseline Model


In [ ]:
results = {}
for name, clf in [('Dummy', DummyClassifier(strategy='most_frequent', random_state=SEED)),
                  ('LogReg', LogisticRegression(max_iter=1000, random_state=SEED)),
                  ('RF', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))]:
    pipe = Pipeline([('pre',preprocessor),('clf',clf)])
    pipe.fit(X_train, y_train); yp = pipe.predict(X_val)
    r = {'Accuracy': accuracy_score(y_val,yp), 'F1': f1_score(y_val,yp,zero_division=0),
         'Recall': recall_score(y_val,yp,zero_division=0)}
    if hasattr(clf, 'predict_proba'):
        try: r['ROC-AUC'] = roc_auc_score(y_val, pipe.predict_proba(X_val)[:,1])
        except: r['ROC-AUC'] = np.nan
    results[name] = r; print(f'{name}: {r}')


## 19. LazyPredict Benchmark


In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)


## 20. FLAML AutoML Run


In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='classification', metric='roc_auc',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'Accuracy': accuracy_score(y_val, yp_fl),
                    'F1': f1_score(y_val, yp_fl, zero_division=0),
                    'ROC-AUC': roc_auc_score(y_val, automl.predict_proba(X_val_lp)[:,1])}
print(results['FLAML'])


## 21. Top 3 Model Selection


In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBClassifier; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3['ExtraTrees'] = ExtraTreesClassifier(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1, eval_metric='logloss')
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3['AdaBoost'] = AdaBoostClassifier(n_estimators=200, random_state=SEED)
top3['GradBoosting'] = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')


## 22. Final Training and Evaluation of Top 3


In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
labels = ['No Purchase', 'Purchased']
final = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t); ypr = mdl.predict_proba(X_te_t)[:,1]
    final[name] = {'Accuracy': accuracy_score(y_test,yp), 'Precision': precision_score(y_test,yp,zero_division=0),
                   'Recall': recall_score(y_test,yp,zero_division=0), 'F1': f1_score(y_test,yp,zero_division=0),
                   'ROC-AUC': roc_auc_score(y_test,ypr)}
    print(f'\n{name}:'); print(classification_report(y_test,yp,target_names=labels))
pd.DataFrame(final).T


In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, m.predict(X_te_t), ax=ax, cmap='Blues', display_labels=labels)
    ax.set_title(n)
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for n,m in top3.items():
    RocCurveDisplay.from_estimator(m, X_te_t, y_test, ax=ax, name=n)
ax.plot([0,1],[0,1],'k--',alpha=0.5); ax.set_title('ROC Curves')
plt.tight_layout(); plt.show()


## 23. Error Analysis


In [ ]:
bm = list(top3.values())[0]; ypb = bm.predict(X_te_t)
fn = (y_test.values==1) & (ypb==0); fp = (y_test.values==0) & (ypb==1)
print(f'False Negatives (missed buyers): {fn.sum()}')
print(f'False Positives (contacted non-buyers): {fp.sum()}')
print(f'Total misclassified: {(y_test.values!=ypb).sum()}/{len(y_test)} ({(y_test.values!=ypb).mean():.1%})')


## 24. Interpretation and Business Insight

- **Vehicle damage history** is a strong predictor
- **Previously insured** customers rarely buy again
- **Age and region** interact with purchase propensity
- **Campaign ROI**: Targeting the top 20% by model score captures majority of buyers


## 25. Limitations

1. Response != actual purchase
2. No temporal dimension
3. Missing competitor pricing
4. Region codes anonymized
5. Single snapshot


## 26. How to Improve This Project

1. Uplift modeling
2. Customer interaction history
3. Probability calibration
4. A/B test model-driven targeting
5. Multi-channel attribution


## 27. Production Considerations

- Batch scoring before campaigns
- Real-time scoring for call center
- Threshold tuning per budget
- Model retraining as products change
- Insurance marketing compliance


## 28. Common Mistakes

1. Using accuracy with 88% majority class
2. Not dropping ID column
3. Ignoring previously insured as leakage risk
4. Not encoding Vehicle_Age ordinally
5. Overfitting on high-cardinality region codes


## 29. Mini Challenge / Exercises

1. Build a lift chart
2. Remove Previously_Insured — AUC impact?
3. Try target encoding for Region_Code
4. Tune threshold for precision > 50%
5. Compare with/without Vehicle_Damage


## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Binary — vehicle insurance cross-sell |
| Dataset | ~380k policyholders (sampled) |
| Difficulty | Medium |
| Key insight | Vehicle damage + no prior insurance = likely buyer |

Response modeling pipeline for insurance cross-sell, business-focused evaluation.
